# 5G Network Intelligence — Single-Source Transfer Experiments

Historical forecasting research. **Default Run All performs no training, calibration or holdout replay.**


## Research question and hypothesis

Does external relative load-shape data reduce original-network WAPE versus the same target-only representation? Compare matched controls; do not attribute a representation change to transfer. A negative result is valid evidence.


## 1. Configuration

Run All is review-only. Set RUN_TRAINING=True deliberately only in a hosted runtime with the source package and data available.


In [ ]:
RUN_TRAINING = False
HORIZONS = (1, 6, 24)
SEED = 17
TARGET_WAPE = 0.10
EXTERNAL_WEIGHT_RATIOS = [0.0, 0.1, 0.5, 1.0]


## Environment and saved evidence

Input: completed worker outputs or the repository. Output: paths to saved reports. Missing files stay unavailable; they are never replaced with fabricated results.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Find report folders in the repo and attached Kaggle outputs.
REPORT_DIRS = [Path('reports'), Path('../reports')]  # Repo root or notebooks/ working directory.
if Path('/kaggle/input').exists():
    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*validation.csv')})

    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*manifest.json')})

def report_path(filename):
    for folder in REPORT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    return None


## 2. Temporal protocol

Original round selection: fit through 419, stop through 503. Refit through 503 with fixed rounds; audit reused validation 504–671. Original final holdout 672–839 stays closed. External data uses only separately prepared chronological prefixes. No fresh-test claim is allowed.


In [ ]:
protocol = pd.DataFrame([{'stage':'round selection fit','last_target':419}, {'stage':'early stopping','last_target':503}, {'stage':'fixed refit','last_target':503}, {'stage':'reused-validation audit','last_target':671}])
display(protocol)


## 3. Preprocessing and representation

The observed 168-hour mean sets a dynamic scale. Only target-history columns are divided by it; phase columns remain deterministic, incompatible exogenous channels are missing for both domains. This compares relative shape without claiming equal physical units.


In [ ]:
# Review the numerical contract without fitting anything.
def relative_target_columns(features, floor=1e-6):
    scale = np.maximum(np.nan_to_num(features[:, 20].astype(float), nan=0.0), floor)
    # Production research implementation resolves this column by FEATURE_NAMES.
    transformed = features.copy()
    transformed[:, :23] /= scale[:, None]
    transformed[:, 25:] = np.nan
    return transformed, scale


## 4. Model, loss and source weights

LightGBM L1 minimizes absolute error. Weighting normalized targets by their observed scale restores original-unit absolute loss. Stop on original-domain MAE; on its fixed actual population, minimizing MAE also minimizes pooled WAPE. The source ratio is predefined.


In [ ]:
# Transparent objective identity; these are arithmetic checks, not model training.
actual = np.array([1.0, 3.0])
point = np.array([0.8, 2.5])
scale = np.array([0.5, 2.0])
np.testing.assert_allclose(np.abs(point / scale - actual / scale) * scale, np.abs(point - actual))
display({'mae': np.abs(point-actual).mean(), 'wape': np.abs(point-actual).sum()/np.abs(actual).sum()})


## 5. Hosted execution

Reusable implementation stays in named source modules instead of a wall of writefile cells. The command below logs progress and saves checkpoints per case/horizon. It is never executed with the default configuration. Review `scripts/train_transfer_pilot.py` or `scripts/train_multisource_pilot.py` for the explicit fit/refit loops.


In [ ]:
if RUN_TRAINING:
    import sys
    import subprocess
    from src.utils.runtime import require_cloud_training
    require_cloud_training()  # Refuses the owner's local Windows runtime.
    subprocess.run([sys.executable, '-u', '-m', 'scripts.train_transfer_pilot'], check=True)
else:
    print('Review mode: no training, calibration or final replay executed.')


## 6. Saved validation results

Units: MAE in original dataset usage, WAPE as a ratio. Multiply WAPE by 100 only for display; never confuse it with MAE. Results remain unavailable until the worker finishes.


In [ ]:
path = report_path('transfer_pilot_validation.csv')
if path:
    results = pd.read_csv(path)
    display(results.loc[results.load_tier == 'All', ['model','horizon','mae','wape','n_valid']])
else:
    print('No completed saved results attached. No scores are invented.')


## Conclusions and debugging

Compare each source case with its representation-matched control at every horizon. Inspect load tiers, coverage and CPU/cloud parity before selecting a candidate. If acquisition, metadata or training fails, inspect the saved log and run ledger; do not reinterpret a failed run as a model result.
